In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch10-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "split.py"

# FreeSolv 的測試分子有多陌生？
第一次需連網下載 MoleculeNet 的原始資料。中間一格把分析寫成暫存程式，最後一格執行並顯示圖。要做小練習，可以把自己的程式接在中間那格最下方，再執行這兩格。

In [ ]:
%%writefile "{script}"
from pathlib import Path
import os
import tempfile
import deepchem as dc
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator

cache = Path(os.environ.get("DEEPCHEM_DATA_DIR", Path(tempfile.gettempdir()) / "cheminfo-datasets"))
cache.mkdir(parents=True, exist_ok=True)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
figure_dir.mkdir(parents=True, exist_ok=True)
generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)

class MorganBits1024(dc.feat.MolecularFeaturizer):
    def _featurize(self, mol, **kwargs):
        return generator.GetFingerprintAsNumPy(mol)

options = dict(featurizer=MorganBits1024(), transformers=[], data_dir=str(cache), save_dir=str(cache))
from collections import Counter
from rdkit import DataStructs
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error, r2_score

tasks, (full,), _ = dc.molnet.load_sampl(splitter=None, **options)
assert tasks == ["expt"] and np.all(full.w > 0)
X, y = full.X, full.y.ravel()
mols = [Chem.MolFromSmiles(s) for s in full.ids]
keys = np.array([MurckoScaffold.MurckoScaffoldSmiles(mol=m) for m in mols])
fps = [generator.GetFingerprint(m) for m in mols]
print("molecules, scaffolds, most frequent:", len(full), len(set(keys)), Counter(keys).most_common(3))

splitters = {"Random": dc.splits.RandomSplitter(), "Scaffold": dc.splits.ScaffoldSplitter()}
fig, axes = plt.subplots(1, 2, figsize=(9, 3.7), layout="constrained")
for i, (name, splitter) in enumerate(splitters.items()):
    train, valid, test = splitter.split(full, seed=0)
    assert len(set(train) | set(valid) | set(test)) == len(full)
    seen = set(keys[train])
    overlap = np.mean([key in seen for key in keys[test]])
    train_fps = [fps[j] for j in train]
    nearest = [max(DataStructs.BulkTanimotoSimilarity(fps[j], train_fps)) for j in test]
    model = RandomForestRegressor(n_estimators=500, random_state=0, n_jobs=1)
    model.fit(X[train], y[train])
    pred = model.predict(X[test])
    print(name, "sizes:", len(train), len(valid), len(test),
          "overlap, median similarity, RMSE, R2:",
          *(f"{v:.3f}" for v in (overlap, np.median(nearest), root_mean_squared_error(y[test], pred),
                                  r2_score(y[test], pred))))
    color = ["#2c6483", "#bc672a"][i]
    axes[0].bar(name, 100 * overlap, color=color)
    axes[0].text(i, 100 * overlap + 2, f"{100*overlap:.1f}%", ha="center")
    axes[1].stairs(np.histogram(nearest, bins=np.linspace(0, 1, 11))[0],
                   np.linspace(0, 1, 11), color=color, label=name, linewidth=2)
axes[0].set(ylabel="Test molecules with a known scaffold (%)", ylim=(0, 110))
axes[1].set(xlabel="Closest training fingerprint (Tanimoto)", ylabel="Test molecules")
axes[1].legend()
fig.savefig(figure_dir / "10-01-freesolv-splits.png", dpi=150)


In [ ]:
from IPython.display import Image, display

subprocess.run([python, str(script)], check=True)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "10-01-freesolv-splits.png")))